# Alignment Faking — Kaggle Runner
GPU: T4×2 (32 GB total, tensor-parallel) or P100 (16 GB single).
Set `TENSOR_PARALLEL = 1` if on P100.

In [ ]:
import os, subprocess, sys, time
from dotenv import load_dotenv

load_dotenv("config.env")

# Hardware overrides only — everything else from config.env
os.environ.setdefault("MODEL_NAME", os.environ["MODEL"])
TENSOR_PARALLEL = 2    # 2 for T4×2, 1 for P100
MAX_MODEL_LEN   = 4096
VLLM_PORT       = int(os.environ.get("VLLM_PORT", 8000))
MODEL           = os.environ["MODEL"]

In [ ]:
# ── Install uv ────────────────────────────────────────────────────────────────
run("curl -LsSf https://astral.sh/uv/install.sh | sh")
os.environ["PATH"] = os.path.expanduser("~/.local/bin") + ":" + os.environ["PATH"]
run("uv --version")

In [ ]:
# ── Clone + sync (uv respects pyproject.toml Python pin + uv.lock) ────────────
if not os.path.isdir(REPO_DIR):
    run(f"git clone {REPO_URL} {REPO_DIR}")
os.chdir(REPO_DIR)
run("uv sync --frozen")

# Add venv to path for this session
venv_bin = os.path.join(REPO_DIR, ".venv", "bin")
os.environ["PATH"] = venv_bin + ":" + os.environ["PATH"]
sys.path.insert(0, os.path.join(REPO_DIR, ".venv", "lib",
                                f"python{sys.version_info.major}.{sys.version_info.minor}",
                                "site-packages"))
print("Env ready")

In [ ]:
# ── HF token (add via Kaggle Secrets: key = HF_TOKEN) ─────────────────────────
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = token
    os.environ["HUGGING_FACE_HUB_TOKEN"] = token
    print("HF token loaded from Kaggle Secrets")
except Exception as e:
    print(f"Kaggle Secrets unavailable ({e}) — falling back to .hf_token file")
    if os.path.exists(".hf_token"):
        with open(".hf_token") as f:
            for line in f:
                k, _, v = line.strip().partition("=")
                os.environ[k.strip()] = v.strip()

In [ ]:
# ── Start vllm server ─────────────────────────────────────────────────────────
import requests

os.makedirs("logs", exist_ok=True)
os.makedirs("data", exist_ok=True)
os.makedirs("evaluated_data", exist_ok=True)

vllm_cmd = (
    f"vllm serve {MODEL}"
    f" --dtype float16"           # T4 doesn't support bfloat16
    f" --gpu-memory-utilization 0.90"
    f" --max-model-len {MAX_MODEL_LEN}"
    f" --tensor-parallel-size {TENSOR_PARALLEL}"
    f" --port {VLLM_PORT}"
)
print(f"Launching: {vllm_cmd}")

log_out = open("logs/vllm.log", "w")
vllm_proc = subprocess.Popen(vllm_cmd, shell=True, stdout=log_out, stderr=log_out)

health_url = f"http://localhost:{VLLM_PORT}/health"
for attempt in range(120):
    try:
        if requests.get(health_url, timeout=2).ok:
            print(f"Server ready (attempt {attempt+1})")
            break
    except Exception:
        pass
    if vllm_proc.poll() is not None:
        raise RuntimeError("vllm process died — check logs/vllm.log")
    time.sleep(5)
else:
    raise RuntimeError("vllm not ready after 10 min")

In [ ]:
# ── Run pipeline ──────────────────────────────────────────────────────────────
try:
    run("python src/gen.py")
    run("python src/eval.py")
finally:
    vllm_proc.terminate()
    log_out.close()
    print("vllm stopped")